In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler

In [14]:
df = pd.read_csv("data/tracks_features.csv")
df = df.drop_duplicates(subset=['id'])
df = df.dropna()

categorical_cols = ['album', 'artists', 'explicit']
numeric_cols = ['danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'duration_ms',
                'time_signature', 'year']

# extract data values and features from dfs
df[numeric_cols] = StandardScaler().fit_transform(df[numeric_cols])

# make categorical data all string types
df[categorical_cols] = df[categorical_cols].astype(str)

# # TODO: working on a subset, do it with whole dataset
# df = df.sample(frac=0.05, random_state=64)

# combine numeric & categorical dfs
X = df[numeric_cols + categorical_cols].to_numpy()
# get indexes of categorical cols
cat_cols_idx = [i for i in range(len(numeric_cols), len(numeric_cols + categorical_cols))]

print(X)
X.shape

[[0.11043044734262245 0.8770996943713197 -1.185890694484223 ...
  'Mortified' '["Dezeray\'s Hammer"]' 'False']
 [-1.0916625328559693 -0.4327801145808334 0.5105957701128729 ...
  'Cold World Plastic Dream' "['Mike Moss']" 'False']
 [-1.7296153863824146 -0.9485877077433393 -1.4686384385837388 ... 'FEZ'
  "['Disasterpeace']" 'False']
 ...
 [-0.6856925351573221 -1.1318351421563349 -0.05489971808615894 ...
  "L'amant confesseur" "['Claude Methé']" 'False']
 [0.21060486235917125 0.7854759771648219 0.22784802601335702 ...
  'The Speed of Light' "['X-Sonic']" 'False']
 [0.7589279761339679 0.7956563901877665 -1.185890694484223 ...
  'Aftermath' "['Cruciform Injection']" 'False']]


(60201, 17)

In [15]:
# choose optimal k
costs = []
for nb_cluster in range(1, 6):
        kprototype = KPrototypes(n_jobs=-1, n_clusters=nb_cluster, random_state=0)
        kprototype.fit_predict(X, categorical=cat_cols_idx)
        costs.append(kprototype.cost_)
        print(f'Nb of cluster: {nb_cluster}, cost: {kprototype.cost_}')

Nb of cluster: 1, cost: 921848.6751869491
Nb of cluster: 2, cost: 768162.9125504


KeyboardInterrupt: 

In [4]:
k_proto = KPrototypes(n_clusters=6, init='Cao', verbose=2, max_iter=20, n_init=1)
clusters = k_proto.fit_predict(X, categorical=cat_cols_idx)
# print(f"center points: {k_proto.cluster_centroids_}")

df['cluster'] = clusters
print(pd.Series(clusters).value_counts())

Initialization method and algorithm are deterministic. Setting n_init to 1.
Init: initializing centroids
Init: initializing clusters
Starting iterations...
Run: 1, iteration: 1/20, moves: 46006, ncost: 1275668.778219017
Run: 1, iteration: 2/20, moves: 18242, ncost: 1262098.0697048968
Run: 1, iteration: 3/20, moves: 9099, ncost: 1257804.8552761972
Run: 1, iteration: 4/20, moves: 6374, ncost: 1254695.8564998799
Run: 1, iteration: 5/20, moves: 5228, ncost: 1252791.4831141715
Run: 1, iteration: 6/20, moves: 4269, ncost: 1251756.1517598014
Run: 1, iteration: 7/20, moves: 3375, ncost: 1251155.2147539693
Run: 1, iteration: 8/20, moves: 2733, ncost: 1250720.7037410203
Run: 1, iteration: 9/20, moves: 2619, ncost: 1250221.3771367867
Run: 1, iteration: 10/20, moves: 3111, ncost: 1249374.7569240984
Run: 1, iteration: 11/20, moves: 4507, ncost: 1247472.1242307993
Run: 1, iteration: 12/20, moves: 6112, ncost: 1244541.7600704718
Run: 1, iteration: 13/20, moves: 6818, ncost: 1240483.9116926077
Run: 1,

In [6]:
def recommend(song_title, df):
    cluster_id = df[df["name"] == song_title]["cluster"].values[0]
    recs = df[df["cluster"] == cluster_id]
    print(recs)


recommend("Freedom", df)


                             id                name               album  \
59       6LiqMOJNu3pMKO6M1cTsgA         Amber Waves      Scarlet's Walk   
77       6RovFMOG0avksTizHRlbUi         Rosaryville         Rosaryville   
87       2z9z18lidxD3euusC3UaLx           Hollywood      New Life Faded   
114      4ZhtIgZ3rIXQnjdDns3I7v        To the Teeth        To The Teeth   
127      2E8AVQcuYSlXm12g112Kef    Untouchable Face              Dilate   
...                         ...                 ...                 ...   
1203822  1l0GWZnfWSNjycuUjp3A3r      La-La-Love You         Sha-La-Love   
1203952  2WX9qJFaeedrpS6ieIA2sx  U R the Best Thing  U R the Best Thing   
1203953  6UnfVMX6VsA5pY7XJAW2iZ         I Just Melt     Christmas Blues   
1203980  1EFqFwCcNYbocZDekC3SyU                Leap                Leap   
1204011  5diStWDmLz9BhHa3io6j2v        Blessed Part     Visionaire - EP   

                       album_id                  artists  \
59       5JnLg2nd43M3NS4NfVrIdQ        